In [3]:
import os

# Check if required file exists
if os.path.exists("go-basic.obo"):
   print("Required file 'go-basic.obo' found in current directory")
else:
   print("Warning: Required file 'go-basic.obo' not found in current directory")
   print("Please add the go-basic.obo file to this directory before proceeding")

Required file 'go-basic.obo' found in current directory


In [10]:
# Quick GO Statistics Analyzer - Read OBO file and show category stats
from collections import defaultdict, Counter

def analyze_go_obo_file():
    """Read and analyze the GO OBO file"""

    # Read the file directly
    filename = 'go-basic.obo'
    print(f"Reading {filename}...")

    with open(filename, 'r', encoding='utf-8') as f:
        content = f.read()

    print(f"File loaded: {len(content):,} characters, {len(content.split())} lines")

    # Parse GO terms
    go_terms = {}
    current_term = None
    lines = content.split('\n')

    for line_num, line in enumerate(lines, 1):
        line = line.strip()

        if line == '[Term]':
            # Save previous term
            if current_term and 'id' in current_term:
                go_terms[current_term['id']] = current_term
            # Start new term
            current_term = {
                'parents': [],
                'part_of_parents': [],
                'relationships': []
            }
        elif current_term:
            if line.startswith('id: GO:'):
                current_term['id'] = line.split(': ')[1]
            elif line.startswith('name: '):
                current_term['name'] = line.split(': ', 1)[1]
            elif line.startswith('namespace: '):
                current_term['namespace'] = line.split(': ')[1]
            elif line.startswith('def: '):
                current_term['definition'] = line.split(': ', 1)[1]
            elif line.startswith('is_a: GO:'):
                parent_id = line.split(': ')[1].split(' !')[0]
                current_term['parents'].append(parent_id)
            elif line.startswith('relationship: part_of GO:'):
                parent_id = line.split('GO:')[1].split(' !')[0]
                parent_id = 'GO:' + parent_id
                current_term['part_of_parents'].append(parent_id)
            elif line.startswith('relationship: '):
                rel_type = line.split('relationship: ')[1].split(' ')[0]
                current_term['relationships'].append(rel_type)
            elif line.startswith('is_obsolete: true'):
                current_term['is_obsolete'] = True

    # Add the last term
    if current_term and 'id' in current_term:
        go_terms[current_term['id']] = current_term

    print(f"Parsed {len(go_terms):,} GO terms")

    # Analyze by category
    print("\n" + "="*80)
    print("GO TERMS BY CATEGORY")
    print("="*80)

    categories = defaultdict(list)
    obsolete_count = 0
    relationship_types = Counter()

    for go_id, term in go_terms.items():
        if term.get('is_obsolete', False):
            obsolete_count += 1
        else:
            namespace = term.get('namespace', 'unknown')
            categories[namespace].append(go_id)

            # Count relationship types
            for rel in term.get('relationships', []):
                relationship_types[rel] += 1

    print("SUMMARY:")
    print(f"  Total terms: {len(go_terms):,}")
    print(f"  Obsolete terms: {obsolete_count:,}")
    print(f"  Active terms: {len(go_terms) - obsolete_count:,}")
    print()

    # Show each category
    for namespace, term_ids in categories.items():
        print(f"{namespace.upper().replace('_', ' ')}: {len(term_ids):,} terms")

        # Find root terms for this category
        root_terms = []
        terms_with_parts = 0

        for go_id in term_ids:
            term = go_terms[go_id]

            # Check if root (no parents)
            if len(term['parents']) == 0 and len(term['part_of_parents']) == 0:
                root_terms.append((go_id, term.get('name', 'No name')))

            # Count part_of relationships
            if len(term['part_of_parents']) > 0:
                terms_with_parts += 1

        print(f"   Root terms: {len(root_terms)}")
        for go_id, name in root_terms:
            print(f"    {go_id}: {name}")

        print(f"  Terms with part_of relationships: {terms_with_parts}")
        print()

    # Show relationship types
    print("RELATIONSHIP TYPES FOUND:")
    for rel_type, count in relationship_types.most_common():
        print(f"  {rel_type}: {count:,} occurrences")

    # Sample terms from each category
    print("\n" + "="*80)
    print("SAMPLE TERMS FROM EACH CATEGORY")
    print("="*80)

    for namespace, term_ids in categories.items():
        if len(term_ids) > 0:
            print(f"\n{namespace.upper().replace('_', ' ')} - Sample 5 terms:")
            for i, go_id in enumerate(term_ids[:5]):
                term = go_terms[go_id]
                name = term.get('name', 'No name')
                parent_count = len(term['parents']) + len(term['part_of_parents'])
                print(f"  {i+1}. {go_id}: {name[:60]}...")
                print(f"     Parents: {parent_count} (is_a: {len(term['parents'])}, part_of: {len(term['part_of_parents'])})")

    print("\nAnalysis complete!")

    return {
        'go_terms': go_terms,
        'categories': categories,
        'total_terms': len(go_terms),
        'active_terms': len(go_terms) - obsolete_count,
        'obsolete_count': obsolete_count
    }

# Run the analysis
result = analyze_go_obo_file()

Reading go-basic.obo...
File loaded: 30,992,530 characters, 3724342 lines
Parsed 47,417 GO terms

GO TERMS BY CATEGORY
SUMMARY:
  Total terms: 47,417
  Obsolete terms: 4,169
  Active terms: 43,248

BIOLOGICAL PROCESS: 27,941 terms
   Root terms: 1
    GO:0008150: biological_process
  Terms with part_of relationships: 4696

MOLECULAR FUNCTION: 11,263 terms
   Root terms: 1
    GO:0003674: molecular_function
  Terms with part_of relationships: 11

CELLULAR COMPONENT: 4,043 terms
   Root terms: 1
    GO:0005575: cellular_component
  Terms with part_of relationships: 1707

EXTERNAL: 1 terms
   Root terms: 0
  Terms with part_of relationships: 0

RELATIONSHIP TYPES FOUND:
  regulates: 3,157 occurrences
  negatively_regulates: 2,729 occurrences
  positively_regulates: 2,718 occurrences

SAMPLE TERMS FROM EACH CATEGORY

BIOLOGICAL PROCESS - Sample 5 terms:
  1. GO:0000001: mitochondrion inheritance...
     Parents: 2 (is_a: 2, part_of: 0)
  2. GO:0000002: mitochondrial genome maintenance...
 

In [ ]:
import json
import pandas as pd
from collections import deque
from multiprocessing import Pool
import os
import matplotlib.pyplot as plt
import networkx as nx

# Global variable to store all complete paths for multiprocessing
ALL_COMPLETE_PATHS = []

def assign_paths_to_term_worker(go_id):
    """Worker function to find all complete paths that pass through a GO term"""
    paths_through_term = []

    for path in ALL_COMPLETE_PATHS:
        if go_id in path:
            paths_through_term.append(path)

    return go_id, paths_through_term

def find_all_complete_paths(category_terms, root_id, n_cores=90):
    """Find all complete paths from root to all leaves, then assign to terms they pass through"""

    global ALL_COMPLETE_PATHS

    print("Finding all complete root-to-leaf paths...")
    print(f"Root: {root_id}")

    # Step 1: Find all complete paths from root to all leaves
    def find_all_root_to_leaf_paths():
        """Find every possible path from root to every leaf"""
        paths = []

        def dfs_to_leaves(current_id, path, visited):
            if current_id in visited:  # Avoid cycles
                return

            visited.add(current_id)

            if current_id in category_terms:
                children = category_terms[current_id]['all_children']

                if not children:  # This is a leaf
                    paths.append(path)
                else:
                    # Continue to all children
                    for child_id in children:
                        dfs_to_leaves(child_id, path + [child_id], visited.copy())

        # Start from root
        dfs_to_leaves(root_id, [root_id], set())
        return paths

    print("Finding all root-to-leaf paths...")
    all_complete_paths = find_all_root_to_leaf_paths()
    print(f"Found {len(all_complete_paths):,} complete paths from root to leaves")

    # Set global variable for multiprocessing
    ALL_COMPLETE_PATHS = all_complete_paths

    # Step 2: For each GO term, find all paths that pass through it
    print("Assigning paths to GO terms...")

    go_ids = list(category_terms.keys())

    print(f"Processing {len(go_ids)} terms with {n_cores} cores...")

    all_term_paths = {}

    # Use smaller batches to avoid memory issues
    batch_size = 500
    total_batches = (len(go_ids) + batch_size - 1) // batch_size

    for batch_idx in range(total_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, len(go_ids))
        batch_go_ids = go_ids[start_idx:end_idx]

        print(f"Processing batch {batch_idx + 1}/{total_batches} ({len(batch_go_ids)} terms)")

        with Pool(n_cores) as pool:
            results = pool.map(assign_paths_to_term_worker, batch_go_ids)

        for go_id, paths in results:
            all_term_paths[go_id] = paths

    print("Path assignment complete")

    return all_term_paths, all_complete_paths

def build_category_tree_structure(category_terms, root_id):
    """Build tree structure with children, parents, siblings, depth"""

    print("Building tree structure...")

    # Initialize children lists
    for go_id, term in category_terms.items():
        term['is_a_children'] = []
        term['part_of_children'] = []
        term['all_children'] = []
        term['all_parents'] = list(set(term.get('parents', []) + term.get('part_of_parents', [])))

    # Build children relationships
    for go_id, term in category_terms.items():
        # Add is_a children
        for parent_id in term.get('parents', []):
            if parent_id in category_terms:
                category_terms[parent_id]['is_a_children'].append(go_id)

        # Add part_of children
        for parent_id in term.get('part_of_parents', []):
            if parent_id in category_terms:
                category_terms[parent_id]['part_of_children'].append(go_id)

    # Combine all children
    for go_id, term in category_terms.items():
        term['all_children'] = list(set(term['is_a_children'] + term['part_of_children']))

    # Calculate depths using BFS
    print("Calculating depths...")
    depths = {root_id: 0}
    queue = deque([root_id])

    while queue:
        current_id = queue.popleft()
        current_depth = depths[current_id]

        if current_id in category_terms:
            for child_id in category_terms[current_id]['all_children']:
                if child_id not in depths:
                    depths[child_id] = current_depth + 1
                    queue.append(child_id)

    # Add depth and leaf info to terms
    for go_id, term in category_terms.items():
        term['depth'] = depths.get(go_id, -1)
        term['is_leaf'] = len(term['all_children']) == 0

    # Find siblings
    print("Finding siblings...")
    for go_id, term in category_terms.items():
        siblings = set()
        for parent_id in term['all_parents']:
            if parent_id in category_terms:
                for sibling_id in category_terms[parent_id]['all_children']:
                    if sibling_id != go_id:
                        siblings.add(sibling_id)
        term['siblings'] = list(siblings)

    print("Tree structure built")
    return category_terms

def visualize_sample_tree(category_terms, root_id, category_name, max_depth=3, max_nodes=30):
    """Visualize a sample of the tree"""

    print(f"Creating {category_name} tree visualization...")

    # Get sample nodes up to max_depth
    sample_nodes = set()
    queue = deque([(root_id, 0)])

    while queue and len(sample_nodes) < max_nodes:
        go_id, depth = queue.popleft()
        if depth <= max_depth and go_id in category_terms:
            sample_nodes.add(go_id)

            # Add some children
            children = category_terms[go_id]['all_children'][:3]  # Limit children
            for child_id in children:
                if len(sample_nodes) < max_nodes:
                    queue.append((child_id, depth + 1))

    # Create networkx graph
    G = nx.DiGraph()

    # Add nodes with labels
    for go_id in sample_nodes:
        if go_id in category_terms:
            term = category_terms[go_id]
            label = f"{go_id}\n{term.get('name', '')[:20]}..."
            depth = term.get('depth', 0)
            is_leaf = term.get('is_leaf', False)

            node_color = 'lightcoral' if is_leaf else 'lightblue'
            if go_id == root_id:
                node_color = 'lightgreen'

            G.add_node(go_id, label=label, depth=depth, color=node_color)

    # Add edges
    for go_id in sample_nodes:
        if go_id in category_terms:
            for child_id in category_terms[go_id]['all_children']:
                if child_id in sample_nodes:
                    G.add_edge(go_id, child_id)

    # Plot
    plt.figure(figsize=(15, 10))

    # Position nodes by depth
    pos = {}
    depth_groups = defaultdict(list)
    for node in G.nodes():
        depth = G.nodes[node]['depth']
        depth_groups[depth].append(node)

    for depth, nodes in depth_groups.items():
        for i, node in enumerate(nodes):
            pos[node] = (i - len(nodes)/2, -depth)

    # Draw
    node_colors = [G.nodes[node]['color'] for node in G.nodes()]
    node_labels = {node: G.nodes[node]['label'] for node in G.nodes()}

    nx.draw(G, pos, with_labels=True, labels=node_labels,
            node_color=node_colors, node_size=3000, font_size=8,
            arrows=True, arrowsize=20, edge_color='gray')

    plt.title(f"{category_name} Tree Sample (Root: {root_id})\nGreen=Root, Blue=Internal, Red=Leaf", fontsize=14)
    plt.tight_layout()
    plt.show()

    print(f"Visualization shows {len(sample_nodes)} nodes up to depth {max_depth}")

def create_final_category_json(category_terms, all_term_paths, category_code):
    """Create final JSON structure for a category"""

    print("Creating final JSON structure...")

    category_analysis = {}

    for go_id, term in category_terms.items():
        category_analysis[go_id] = {
            'id': go_id,
            'name': term.get('name', ''),
            'definition': term.get('definition', ''),
            'category': category_code,
            'parents': term['all_parents'],
            'children': term['all_children'],
            'siblings': term['siblings'],
            'all_paths_through_term': all_term_paths.get(go_id, []),
            'depth': term['depth'],
            'is_leaf': term['is_leaf'],
            'relationship_types': {
                'is_a_parents': term.get('parents', []),
                'part_of_parents': term.get('part_of_parents', []),
                'is_a_children': term['is_a_children'],
                'part_of_children': term['part_of_children']
            }
        }

    return category_analysis

def process_single_category(go_terms, category_code, namespace, root_id, n_cores=90, visualize=True):
    """Process a single GO category with complete root-to-leaf paths"""

    print(f"\n{'='*80}")
    print(f"PROCESSING {category_code} ({namespace})")
    print(f"{'='*80}")

    # Filter terms for this category
    category_terms = {go_id: term for go_id, term in go_terms.items()
                     if term.get('namespace') == namespace and not term.get('is_obsolete', False)}

    print(f"Processing {len(category_terms):,} {category_code} terms")

    if len(category_terms) == 0:
        print(f"No terms found for {category_code}")
        return None, None

    # Build tree structure
    category_terms = build_category_tree_structure(category_terms, root_id)

    # Visualize if requested
    if visualize:
        visualize_sample_tree(category_terms, root_id, category_code)

    # Find all complete paths
    all_term_paths, all_complete_paths = find_all_complete_paths(category_terms, root_id, n_cores)

    # Create final JSON structure
    category_analysis = create_final_category_json(category_terms, all_term_paths, category_code)

    # Save individual category file
    output_file = f'{category_code.lower()}_complete_paths_analysis.json'
    with open(output_file, 'w') as f:
        json.dump(category_analysis, f, indent=2)

    print(f"Saved {category_code} analysis to {output_file}")

    # Show statistics
    print(f"\n{category_code} COMPLETE PATHS ANALYSIS STATS:")
    print(f"{'='*60}")
    total_terms = len(category_analysis)
    leaf_terms = sum(1 for term in category_analysis.values() if term['is_leaf'])
    max_depth = max(term['depth'] for term in category_analysis.values()) if category_analysis else 0
    total_complete_paths = len(all_complete_paths)
    total_assigned_paths = sum(len(term['all_paths_through_term']) for term in category_analysis.values())

    print(f"Total {category_code} terms: {total_terms:,}")
    print(f"Leaf terms: {leaf_terms:,} ({leaf_terms/total_terms*100:.1f}%)")
    print(f"Maximum depth: {max_depth}")
    print(f"Total complete root-to-leaf paths: {total_complete_paths:,}")
    print(f"Total assigned path instances: {total_assigned_paths:,}")
    print(f"Average paths per term: {total_assigned_paths/total_terms:.1f}")

    # Show sample analysis with complete paths
    print(f"\nSAMPLE {category_code} COMPLETE PATHS:")
    print(f"{'='*60}")

    # Find a mid-level term for good example
    sample_terms = [term for term in category_analysis.values()
                   if term['depth'] >= 2 and len(term['all_paths_through_term']) > 0]

    if sample_terms:
        sample = sample_terms[0]
        print(f"Sample term: {sample['id']} - {sample['name']}")
        print(f"Depth: {sample['depth']}, Paths through term: {len(sample['all_paths_through_term'])}")
        print("\nFirst 3 complete paths:")
        for i, path in enumerate(sample['all_paths_through_term'][:3]):
            print(f"  Path {i+1}: {' → '.join(path)}")
            print(f"    Length: {len(path)} terms")

    return category_analysis, all_complete_paths

def process_all_go_categories(go_terms, n_cores=90):
    """Process all GO categories (CC, MF, BP) with complete root-to-leaf paths"""

    print("PROCESSING ALL GO CATEGORIES WITH COMPLETE PATHS")
    print("="*80)

    categories = {
        'CC': ('cellular_component', 'GO:0005575'),
        'MF': ('molecular_function', 'GO:0003674'),
        'BP': ('biological_process', 'GO:0008150')
    }

    all_results = {}
    all_paths_data = {}

    # Process each category
    for category_code, (namespace, root_id) in categories.items():

        # Only visualize for CC (first one)
        visualize = (category_code == 'CC')

        category_analysis, all_complete_paths = process_single_category(
            go_terms, category_code, namespace, root_id, n_cores, visualize
        )

        if category_analysis:
            all_results[category_code] = category_analysis
            all_paths_data[category_code] = all_complete_paths

    # Save combined results
    print(f"\n{'='*80}")
    print("SAVING COMBINED RESULTS")
    print("="*80)

    combined_output_file = 'all_go_categories_complete_paths.json'
    with open(combined_output_file, 'w') as f:
        json.dump(all_results, f, indent=2)

    print(f"Saved combined analysis to {combined_output_file}")

    # Show overall statistics
    print("\nOVERALL STATISTICS:")
    print(f"{'='*50}")

    total_terms = 0
    total_paths = 0

    for category_code, analysis in all_results.items():
        terms_count = len(analysis)
        paths_count = len(all_paths_data[category_code])
        total_terms += terms_count
        total_paths += paths_count

        print(f"{category_code}: {terms_count:,} terms, {paths_count:,} complete paths")

    print(f"\nCOMBINED: {total_terms:,} total terms, {total_paths:,} total complete paths")

    print("\nAll categories processing complete")
    print("Ready for LLM training with complete semantic paths")

    return all_results, all_paths_data

# Main execution
print("STARTING COMPLETE GO TREE ANALYSIS")
print("="*80)

# Process all categories
all_go_analysis, all_paths_data = process_all_go_categories(result['go_terms'], n_cores=90)

print("\nAnalysis complete")
print("Files generated:")
print("  - cc_complete_paths_analysis.json")
print("  - mf_complete_paths_analysis.json")
print("  - bp_complete_paths_analysis.json")
print("  - all_go_categories_complete_paths.json")

In [ ]:
# Fixing the data to string, remove citations from definitions, add language to answers and to not available answers
import random
import re

# Your refined question templates
QUESTION_TEMPLATES = {
   "category": [
       "What GO category does {go_id} belong to?",
       "Which GO category is {go_id} in?",
       "Is {go_id} a molecular function, biological process, or cellular component?",
       "What type of GO term is {go_id}?",
       "Which GO ontology branch contains {go_id}?",
       "What is the GO aspect of {go_id}?",
       "How is {go_id} classified in the Gene Ontology?",
       "What GO namespace does {go_id} belong to?",
       "Which of the three main GO categories contains {go_id}?",
       "What is the GO category classification of {go_id}?"
   ],
   "name": [
       "What is the name of {go_id}?",
       "What is {go_id} called?",
       "What is the official term for {go_id}?",
       "What name is assigned to {go_id}?",
       "What is the GO term name for {go_id}?",
       "What is {go_id} known as?",
       "What is the term name of {go_id}?",
       "What is the official name of {go_id}?",
       "What name corresponds to {go_id}?",
       "What is the full name of {go_id}?"
   ],
   "definition": [
       "What is the definition of {go_id}?",
       "How is {go_id} defined?",
       "Can you explain what {go_id} is?",
       "What does the term {go_id} describe?",
       "Please describe {go_id}.",
       "What is the scientific definition of {go_id}?",
       "Can you provide the definition for {go_id}?",
       "What is the detailed explanation of {go_id}?",
       "What is the formal definition of {go_id}?",
       "What is the meaning and description of {go_id}?"
   ],
   "parents": [
       "What are the parents of {go_id}?",
       "What are the parent terms of {go_id}?",
       "Which terms are parents of {go_id}?",
       "What are the immediate ancestors of {go_id}?",
       "What terms is {go_id} a child of?",
       "Which terms is {go_id} under?",
       "What are the direct parents of {go_id}?",
       "What terms does {go_id} descend from?",
       "What are the immediate parent terms of {go_id}?",
       "Which GO terms are direct parents of {go_id}?"
   ],
   "children": [
       "What are the children of {go_id}?",
       "What are the child terms of {go_id}?",
       "Which terms are children of {go_id}?",
       "What are the immediate descendants of {go_id}?",
       "What terms is {go_id} a parent of?",
       "What are the direct children of {go_id}?",
       "What subterms does {go_id} have?",
       "What terms descend from {go_id}?",
       "What are the immediate child terms of {go_id}?",
       "Which GO terms are direct children of {go_id}?"
   ],
   "siblings": [
       "What are the siblings of {go_id}?",
       "What are the sibling terms of {go_id}?",
       "Which terms are siblings of {go_id}?",
       "What terms share the same parents as {go_id}?",
       "What terms share parents with {go_id}?",
       "What other children do {go_id}'s parents have?",
       "What terms are co-children with {go_id}?",
       "What terms have identical parents to {go_id}?",
       "Which terms are sibling nodes to {go_id}?",
       "What are the other child terms of {go_id}'s parents?"
   ],
   "paths": [
       "What are all the paths that go through {go_id}?",
       "What are the complete paths containing {go_id}?",
       "List all paths that include {go_id}.",
       "What are the root-to-leaf paths through {go_id}?",
       "Show me all paths passing through {go_id}.",
       "What are the full hierarchical paths for {go_id}?",
       "Which paths contain {go_id}?",
       "What are the end-to-end paths through {go_id}?",
       "What are all the complete pathways that include {go_id}?",
       "What are the full ontology paths containing {go_id}?"
   ]
}

# Response templates for when data is available
RESPONSE_TEMPLATES_AVAILABLE = {
   "category": [
       "The GO category of {go_id} is {category}.",
       "{go_id} belongs to the {category} category.",
       "{go_id} is classified as {category}.",
       "The ontology branch for {go_id} is {category}.",
       "{go_id} is categorized under {category}."
   ],
   "name": [
       "The name of {go_id} is {name}.",
       "{go_id} is called {name}.",
       "The GO term name for {go_id} is {name}.",
       "{go_id} is named {name}.",
       "The official name of {go_id} is {name}."
   ],
   "definition": [
       "The definition of {go_id} is: {definition}",
       "{go_id} is defined as: {definition}",
       "{go_id} means: {definition}",
       "The scientific definition of {go_id} is: {definition}",
       "{go_id} is described as: {definition}"
   ],
   "parents": [
       "The parents of {go_id} are: {parents_list}.",
       "The parent terms of {go_id} are: {parents_list}.",
       "{go_id} has the following parents: {parents_list}.",
       "The direct parents of {go_id} are: {parents_list}.",
       "{go_id} descends from: {parents_list}."
   ],
   "children": [
       "The children of {go_id} are: {children_list}.",
       "The child terms of {go_id} are: {children_list}.",
       "{go_id} has the following children: {children_list}.",
       "The direct children of {go_id} are: {children_list}.",
       "{go_id} is a parent of: {children_list}."
   ],
   "siblings": [
       "The siblings of {go_id} are: {siblings_list}.",
       "The sibling terms of {go_id} are: {siblings_list}.",
       "{go_id} has the following siblings: {siblings_list}.",
       "Terms that share parents with {go_id} are: {siblings_list}.",
       "The sibling nodes of {go_id} are: {siblings_list}."
   ],
   "paths": [
       "The complete paths through {go_id} are: {paths_list}.",
       "All paths containing {go_id} are: {paths_list}.",
       "{go_id} appears in the following paths: {paths_list}.",
       "The root-to-leaf paths for {go_id} are: {paths_list}.",
       "The hierarchical paths through {go_id} are: {paths_list}."
   ]
}

# Response templates for when data is not available
RESPONSE_TEMPLATES_NOT_AVAILABLE = {
   "category": [
       "{go_id} has no category information.",
       "No category found for {go_id}.",
       "The category for {go_id} is not available.",
       "Category information for {go_id} is missing.",
       "{go_id} does not have a defined category."
   ],
   "name": [
       "{go_id} has no name information.",
       "No name found for {go_id}.",
       "The name for {go_id} is not available.",
       "Name information for {go_id} is missing.",
       "{go_id} does not have a defined name."
   ],
   "definition": [
       "{go_id} has no definition available.",
       "No definition found for {go_id}.",
       "The definition for {go_id} is not available.",
       "Definition information for {go_id} is missing.",
       "{go_id} does not have a defined description."
   ],
   "parents": [
       "{go_id} has no parents.",
       "There are no parent terms for {go_id}.",
       "{go_id} is a root node with no parents.",
       "No parents found for {go_id}.",
       "{go_id} does not have any parent terms."
   ],
   "children": [
       "{go_id} has no children.",
       "There are no child terms for {go_id}.",
       "{go_id} is a leaf node with no children.",
       "No children found for {go_id}.",
       "{go_id} does not have any child terms."
   ],
   "siblings": [
       "{go_id} has no siblings.",
       "There are no sibling terms for {go_id}.",
       "{go_id} is an only child with no siblings.",
       "No siblings found for {go_id}.",
       "{go_id} does not have any sibling terms."
   ],
   "paths": [
       "{go_id} has no paths available.",
       "No paths found for {go_id}.",
       "There are no complete paths containing {go_id}.",
       "Path information for {go_id} is not available.",
       "{go_id} does not appear in any complete paths."
   ]
}

def clean_definition(definition):
    """Clean GO definition by removing citations and extra quotes"""
    if not definition:
        return ""

    # Convert to string if not already
    definition = str(definition)

    # Remove citations patterns FIRST (before quote removal)
    citation_patterns = [
        r'\[PMID:\d+\]',
        r'\[GOC:[^\]]+\]',
        r'\[EC:[^\]]+\]',
        r'\[ISBN:[^\]]+\]',
        r'\[DOI:[^\]]+\]',
        r'\[TC:[^\]]+\]',
        r'\[RHEA:[^\]]+\]',
        r'\[MetaCyc:[^\]]+\]',
        r'\[KEGG:[^\]]+\]',
        r'\[UniProtKB:[^\]]+\]'
    ]

    for pattern in citation_patterns:
        definition = re.sub(pattern, '', definition)

    # Remove any remaining bracket patterns like [something]
    definition = re.sub(r'\[[^\]]*\]', '', definition)

    # Remove ALL types of quotes more aggressively
    definition = definition.strip()

    # Remove quotes from beginning and end repeatedly
    while True:
        old_def = definition
        # Remove outer quotes
        if definition.startswith('"') and definition.endswith('"'):
            definition = definition[1:-1].strip()
        elif definition.startswith("'") and definition.endswith("'"):
            definition = definition[1:-1].strip()
        # If no change, break the loop
        if definition == old_def:
            break

    # Remove any remaining quotes that might be wrapping the entire definition
    definition = definition.strip()
    # Fixed regex - use raw string with escaped quotes
    definition = re.sub(r'^["\'](.+)["\']$', r'\1', definition)

    # Remove any stray quotes at the beginning or end
    definition = definition.strip('"').strip("'").strip()

    # Clean up extra spaces and periods
    definition = re.sub(r'\s+', ' ', definition)  # Multiple spaces to single space
    definition = re.sub(r'\s*\.\s*$', '.', definition)  # Fix ending period
    definition = definition.strip()

    return definition

def format_list_for_response(data_list):
   """Convert list to comma-separated string for easier training comparison"""
   if not data_list:
       return ""

   # Simple comma separation - no "and" for better training
   return ", ".join(data_list)

def format_paths_for_response(paths_list):
    """Format paths list for response"""
    if not paths_list:
        return ""

    formatted_paths = []
    for path in paths_list:
        if isinstance(path, list):
            path_str = ", ".join(path)
            formatted_paths.append(f"[{path_str}]")
        else:
            formatted_paths.append(f"[{str(path)}]")

    return "; ".join(formatted_paths)

def generate_refined_qa_pair(go_id, go_data, question_type, used_templates):
   """Generate a single refined Q&A pair"""

   # Sample question template
   available_templates = [t for t in QUESTION_TEMPLATES[question_type] if t not in used_templates[question_type]]
   if not available_templates:
       used_templates[question_type] = set()
       available_templates = QUESTION_TEMPLATES[question_type]

   question_template = random.choice(available_templates)
   used_templates[question_type].add(question_template)
   question = question_template.format(go_id=go_id)

   # Get data and check availability
   if question_type == 'category':
       category_map = {
           'CC': 'Cellular Component',
           'MF': 'Molecular Function',
           'BP': 'Biological Process'
       }
       data = category_map.get(go_data['category'], go_data['category'])
       is_available = bool(data)

       if is_available:
           response_template = random.choice(RESPONSE_TEMPLATES_AVAILABLE[question_type])
           answer = response_template.format(go_id=go_id, category=data)
       else:
           response_template = random.choice(RESPONSE_TEMPLATES_NOT_AVAILABLE[question_type])
           answer = response_template.format(go_id=go_id)

   elif question_type == 'name':
       data = go_data.get('name', '')
       is_available = bool(data)

       if is_available:
           response_template = random.choice(RESPONSE_TEMPLATES_AVAILABLE[question_type])
           answer = response_template.format(go_id=go_id, name=data)
       else:
           response_template = random.choice(RESPONSE_TEMPLATES_NOT_AVAILABLE[question_type])
           answer = response_template.format(go_id=go_id)

   elif question_type == 'definition':
       data = clean_definition(go_data.get('definition', ''))
       is_available = bool(data)

       if is_available:
           response_template = random.choice(RESPONSE_TEMPLATES_AVAILABLE[question_type])
           answer = response_template.format(go_id=go_id, definition=data)
       else:
           response_template = random.choice(RESPONSE_TEMPLATES_NOT_AVAILABLE[question_type])
           answer = response_template.format(go_id=go_id)

   elif question_type in ['parents', 'children', 'siblings']:
       data_list = go_data.get(question_type, [])
       is_available = bool(data_list)

       if is_available:
           formatted_list = format_list_for_response(data_list)
           response_template = random.choice(RESPONSE_TEMPLATES_AVAILABLE[question_type])
           answer = response_template.format(go_id=go_id, **{f"{question_type}_list": formatted_list})
       else:
           response_template = random.choice(RESPONSE_TEMPLATES_NOT_AVAILABLE[question_type])
           answer = response_template.format(go_id=go_id)

   elif question_type == 'paths':
       data_list = go_data.get('all_paths_through_term', [])
       is_available = bool(data_list)

       if is_available:
           formatted_paths = format_paths_for_response(data_list)
           response_template = random.choice(RESPONSE_TEMPLATES_AVAILABLE[question_type])
           answer = response_template.format(go_id=go_id, paths_list=formatted_paths)
       else:
           response_template = random.choice(RESPONSE_TEMPLATES_NOT_AVAILABLE[question_type])
           answer = response_template.format(go_id=go_id)

   return {
       'question': question,
       'answer': answer,
       'go_id': go_id,
       'go_name': go_data.get('name', ''),
       'go_category': go_data.get('category', ''),
       'question_type': question_type,
       'has_data': is_available
   }

def load_go_data():
   """Load GO data from existing JSON"""
   print("Loading GO analysis data...")

   with open('cc_complete_paths_analysis.json', 'r') as f:
       cc_data = json.load(f)
   with open('mf_complete_paths_analysis.json', 'r') as f:
       mf_data = json.load(f)
   with open('bp_complete_paths_analysis.json', 'r') as f:
       bp_data = json.load(f)

   print(f"Loaded: CC({len(cc_data)}), MF({len(mf_data)}), BP({len(bp_data)})")
   return {'CC': cc_data, 'MF': mf_data, 'BP': bp_data}

def generate_refined_training_data(go_data_dict):
   """Generate refined training data with templates"""
   print("Generating refined training data...")

   all_qa_pairs = []
   categories_order = ['CC', 'MF', 'BP']
   question_types = ['category', 'name', 'definition', 'parents', 'children', 'siblings', 'paths']

   # Track used templates
   used_templates = {q_type: set() for q_type in question_types}

   for category in categories_order:
       category_data = go_data_dict[category]
       print(f"\nProcessing {category} category ({len(category_data)} terms)...")

       for i, (go_id, go_data) in enumerate(category_data.items()):
           if i % 1000 == 0:
               print(f"   Processed {i}/{len(category_data)} terms...")

           # Generate 7 Q&A pairs for this GO term
           for question_type in question_types:
               qa_pair = generate_refined_qa_pair(go_id, go_data, question_type, used_templates)
               all_qa_pairs.append(qa_pair)

       print(f"Generated Q&A pairs for {category}")

   print(f"\nTotal refined Q&A pairs generated: {len(all_qa_pairs)}")
   return all_qa_pairs

def save_refined_data(qa_pairs):
   """Save refined data"""
   print("\nSaving refined training data...")

   # Save as JSON
   json_file = 'go_qa_refined_dataset.json'
   with open(json_file, 'w') as f:
       json.dump(qa_pairs, f, indent=2)
   print(f"Saved JSON: {json_file}")

   # Create DataFrame and save as Parquet
   df = pd.DataFrame(qa_pairs)
   parquet_file = 'go_qa_refined_dataset.parquet'
   df.to_parquet(parquet_file, index=False)
   print(f"Saved Parquet: {parquet_file}")

   # Show statistics
   print("\nREFINED DATASET STATS:")
   print(f"Total examples: {len(df):,}")
   print(f"Categories: {df['go_category'].value_counts().to_dict()}")
   print(f"Question types: {df['question_type'].value_counts().to_dict()}")
   print(f"Has data: {df['has_data'].value_counts().to_dict()}")

   # Show samples
   print("\nSAMPLE REFINED DATA:")
   for i in range(3):
       row = df.iloc[i]
       print(f"\nSample {i+1}:")
       print(f"  Question: {row['question']}")
       print(f"  Answer: {row['answer']}")
       print(f"  Has Data: {row['has_data']}")
       print(f"  Question Type: {row['question_type']}")

   return df

# Main execution
def main():
   print("GENERATING REFINED GO Q&A DATASET")
   print("="*60)

   random.seed(42)
   go_data = load_go_data()
   qa_pairs = generate_refined_training_data(go_data)
   df = save_refined_data(qa_pairs)

   print("\nREFINED DATASET COMPLETE")
   print("Files created:")
   print("  - go_qa_refined_dataset.json")
   print("  - go_qa_refined_dataset.parquet")

   return qa_pairs, df

# Run the refined generation
if __name__ == "__main__":
   qa_pairs, df = main()

In [11]:
# Data validation check

def validate_dataset():
   """Quick validation of the refined GO Q&A dataset"""

   print("DATASET VALIDATION")
   print("="*40)

   # Load data
   df = pd.read_parquet('go_qa_refined_dataset.parquet')

   # Basic stats
   print(f"Total examples: {len(df):,}")
   print(f"Columns: {list(df.columns)}")

   # Data type validation
   print("\nDATA TYPES:")
   for col in df.columns:
       print(f"  {col}: {df[col].dtype}")

   # String validation for key columns
   print("\nSTRING VALIDATION:")
   question_strings = df['question'].apply(lambda x: isinstance(x, str)).all()
   answer_strings = df['answer'].apply(lambda x: isinstance(x, str)).all()
   print(f"  All questions are strings: {question_strings}")
   print(f"  All answers are strings: {answer_strings}")

   # Missing values
   print("\nMISSING VALUES:")
   missing = df.isnull().sum()
   for col, count in missing.items():
       print(f"  {col}: {count}")

   # Distribution check
   print("\nDISTRIBUTION:")
   print(f"  Categories: {df['go_category'].value_counts().to_dict()}")
   print(f"  Question types: {df['question_type'].value_counts().to_dict()}")
   print(f"  Has data: {df['has_data'].value_counts().to_dict()}")

   # Answer length issues
   print("\nANSWER LENGTH CHECK:")
   long_answers = (df['answer'].str.len() > 10000).sum()
   max_length = df['answer'].str.len().max()
   print(f"  Answers >10k chars: {long_answers}")
   print(f"  Max answer length: {max_length:,}")

   # Final status
   print("\nVALIDATION STATUS:")
   issues = []
   if not question_strings: issues.append("Non-string questions")
   if not answer_strings: issues.append("Non-string answers")
   if missing.sum() > 0: issues.append("Missing values")
   if long_answers > 0: issues.append("Extremely long answers")

   if issues:
       print(f"  Issues found: {', '.join(issues)}")
   else:
       print("  Status: All checks passed")

   return df

# Run validation
df = validate_dataset()

DATASET VALIDATION
Total examples: 302,729
Columns: ['question', 'answer', 'go_id', 'go_name', 'go_category', 'question_type', 'has_data']

DATA TYPES:
  question: object
  answer: object
  go_id: object
  go_name: object
  go_category: object
  question_type: object
  has_data: bool

STRING VALIDATION:
  All questions are strings: True
  All answers are strings: True

MISSING VALUES:
  question: 0
  answer: 0
  go_id: 0
  go_name: 0
  go_category: 0
  question_type: 0
  has_data: 0

DISTRIBUTION:
  Categories: {'BP': 195587, 'MF': 78841, 'CC': 28301}
  Question types: {'category': 43247, 'name': 43247, 'definition': 43247, 'parents': 43247, 'children': 43247, 'siblings': 43247, 'paths': 43247}
  Has data: {True: 276546, False: 26183}

ANSWER LENGTH CHECK:
  Answers >10k chars: 4913
  Max answer length: 55,311,552

VALIDATION STATUS:
  Issues found: Extremely long answers
